### RAG end to end pipeline ###

In [3]:
import tqdm
import os
from pathlib import Path
from langchain_community import document_loaders
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

C:\Users\admin\AppData\Local\Temp\ipykernel_5460\838187882.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community import document_loaders
c:\Users\admin\Documents\Personal_documents\Projects\Research-Paper-Brain-RAG-\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Function to process PDF documents and extract text
def process_documents(doc_dir):
    doc_dir = "../data"
    all_documents = []
    pdf_directory = Path(doc_dir)
    pdf_files = list(pdf_directory.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files in {pdf_directory}")

    for pdf in pdf_files:
        print(f"Processing {pdf}...")

        try:
            loader = PyPDFLoader(str(pdf))
            documents = loader.load()

            ##adding the metadata to the documents

            for doc in documents:
                doc.metadata["source"] = pdf.name

            all_documents.extend(documents)
        except Exception as e:
            print(f"Error processing {pdf}: {e}")   

    print(f"Total documents processed: {len(all_documents)}")
    return all_documents  


In [5]:
### text splitting the documents into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    all_chunks = []
    for doc in documents:
        chunks = text_splitter.split_documents([doc])
        all_chunks.extend(chunks)

    print(f"Total chunks created: {len(all_chunks)}")
    return all_chunks

all_documents = process_documents("../data")
all_chunks = split_documents(all_documents)

Found 2 PDF files in ..\data
Processing ..\data\attention.pdf...
Processing ..\data\research_paper.pdf...
Total documents processed: 42
Total chunks created: 168


### Embedding and VectorStore DB ###

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [7]:
class EmbeddingManager:
    
    def __init__ (self, model_name: str = "all-MiniLM-L6-v2"):

        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):    
        try:

            print(f"Loading embedding model: {self.model_name}...")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")

        except Exception as e:
            
            print(f"Error loading model {self.model_name}: {e}")
            raise e

    def generate_embeddings (self, texts: List[str]) -> np.ndarray:
        try:
            embeddings = self.model.encode(texts, convert_to_numpy=True, show_progress_bar=True)
            return embeddings
        except Exception as e:
            print(f"Error generating embeddings: {e}")
            raise e

embedding_manager = EmbeddingManager()
embedding_manager
print("Embeddings generated successfully.")

Loading embedding model: all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6188.42it/s]


Model loaded successfully. Embedding dimension: 384
Embeddings generated successfully.


In [8]:
class VectorStore:

    def __init__(self, collection_name: str = "pdf_collection", persist_directory: str = "../data/vectorestore/chroma_db"):

    
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):

        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "Collection of PDF document embeddings for RAG"})
            print(f"Vector store initialized at {self.persist_directory} with collection '{self.collection_name}'.")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise e

    def add_embeddings(self, documents: List[Any], embeddings: np.ndarray):

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must match.")


        ids =[]
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = str(uuid.uuid4())
            ids.append(doc_id)
            metadatas.append(doc.metadata)
            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())


        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=documents_text,
                embeddings=embeddings_list
            )
            print(f"Added {len(documents)} documents to the vector store.")

        except Exception as e:
            print(f"Error adding embeddings to vector store: {e}")
            raise e    

vector_store = VectorStore()
vector_store

Vector store initialized at ../data/vectorestore/chroma_db with collection 'pdf_collection'.


In [9]:
text =[doc.page_content for doc in all_chunks]

embeddings = embedding_manager.generate_embeddings(text)

vector_store.add_embeddings(all_chunks, embeddings)


Batches: 100%|██████████| 6/6 [00:06<00:00,  1.14s/it]


Added 168 documents to the vector store.


In [10]:
class RagRetriever:

    def __init__(self, vector_store:VectorStore, embedding_manager: EmbeddingManager):
        
        self.vector_store = vector_store
        self.embedding_manager= embedding_manager

    def retrieve (self, query: str, top_k: int = 5, similarity_threshold: float = 0.0) -> List[Dict[str, Any]]:

        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
                include=["metadatas", "documents", "embeddings"]
            )

            retrieved_docs = []
            for i in range(len(results["ids"][0])):
                doc_id = results["ids"][0][i]
                metadata = results["metadatas"][0][i]
                document_text = results["documents"][0][i]
                embedding = np.array(results["embeddings"][0][i])

                similarity_score = cosine_similarity([query_embedding], [embedding])[0][0]

                if similarity_score >= similarity_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "metadata": metadata,
                        "document": document_text,
                        "similarity_score": similarity_score
                    })

            return retrieved_docs
        except Exception as e:
            print(f"Error occurred while retrieving documents: {e}")
            return []


In [17]:
rag_retriever=RagRetriever(vector_store=vector_store, embedding_manager=embedding_manager)
rag_retriever.retrieve(query="what is weather?")

Batches: 100%|██████████| 1/1 [00:00<00:00, 77.41it/s]


[{'id': '63654502-104f-4ccc-aad7-985a10a8ac8c',
  'metadata': {'title': 'LLM-as-a-Verifier: A General-Purpose Verification Framework',
   'trapped': '/False',
   'creationdate': '',
   'author': 'Jacky Kwok; Shulu Li; Pranav Atreya; Yuejiang Liu; Yixing Jiang; Chelsea Finn; Marco Pavone; Ion Stoica; Azalia Mirhoseini',
   'page': 20,
   'producer': 'pikepdf 8.15.1',
   'creator': 'arXiv GenPDF (tex2pdf:a6404ea)',
   'source': 'research_paper.pdf',
   'doi': 'https://doi.org/10.48550/arXiv.2607.05391',
   'arxivid': 'https://arxiv.org/abs/2607.05391v1',
   'page_label': '21',
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1',
   'total_pages': 31,
   'license': 'http://creativecommons.org/licenses/by/4.0/'},
  'document': 'URLhttps://arxiv.org/abs/2302.04166.\n[44] Yang Liu, Dan Iter, Yichong Xu, Shuohang Wang, Ruochen Xu, and Chenguang Zhu. G-eval:\nNLG evaluation using GPT-4 with better human alignment. InProceedings of the 